Praktikum 3 - Klasifikasi teks SMS spam/ham dengan Multinomial Naive Bayes.

In [1]:
# Shim: notebook tidak punya __file__, kode ini warisan script .py.
import os
__file__ = os.path.abspath("naive_bayes_spam.ipynb")
print("base:", os.path.dirname(__file__))

base: C:\code\pemmes26\js09-klasifikasi\praktikum-3


In [2]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
OUT = HERE / "output"
OUT.mkdir(exist_ok=True)

In [3]:
# Langkah 1 - Load data (latin-1 sebagai fallback karena data asli tidak UTF-8)
path = ROOT / "data" / "spam.csv"
try:
    df = pd.read_csv(path, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(path, encoding="latin-1")
print("=== Langkah 1: Load Data ===")
print(df.head())

=== Langkah 1: Load Data ===
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [4]:
# Langkah 2a - Drop kolom tidak terpakai, rename kolom
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df = df.rename(columns={"v1": "Labels", "v2": "SMS"})
df = df[["Labels", "SMS"]].dropna()

In [5]:
# Langkah 2b - Inspeksi
print("\n=== Langkah 2b: Inspeksi ===")
print(df["Labels"].value_counts())
print(df.info())
print(df.describe())


=== Langkah 2b: Inspeksi ===
Labels
ham     4825
spam     747
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Labels  5572 non-null   object
 1   SMS     5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None
       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


In [6]:
# Langkah 2c - Encoding label: spam=1, ham=0
df["Labels"] = df["Labels"].map({"spam": 1, "ham": 0})
print("\n=== Langkah 2c: Setelah Encoding ===")
print(df.head())


=== Langkah 2c: Setelah Encoding ===
   Labels                                                SMS
0       0  Go until jurong point, crazy.. Available only ...
1       0                      Ok lar... Joking wif u oni...
2       1  Free entry in 2 a wkly comp to win FA Cup fina...
3       0  U dun say so early hor... U c already then say...
4       0  Nah I don't think he goes to usf, he lives aro...


In [7]:
# Langkah 2d - Pisah fitur dan label
X = df["SMS"].values
y = df["Labels"].values

In [8]:
# Langkah 3 - Split dulu, baru ekstraksi fitur (hindari information leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50)
bow = CountVectorizer()
Xtr = bow.fit_transform(X_train)
Xte = bow.transform(X_test)
print("\n=== Langkah 3: Ekstraksi Fitur ===")
print("Jumlah fitur:", len(bow.get_feature_names_out()))
print(f"Dimensi data train: {Xtr.shape}, test: {Xte.shape}")


=== Langkah 3: Ekstraksi Fitur ===
Jumlah fitur: 7727
Dimensi data train: (4457, 7727), test: (1115, 7727)


In [9]:
# Langkah 4 - Training dan evaluasi
mnb = MultinomialNB().fit(Xtr, y_train)
acc_train = accuracy_score(y_train, mnb.predict(Xtr))
y_pred = mnb.predict(Xte)
acc_test = accuracy_score(y_test, y_pred)
print("\n=== Langkah 4: Evaluasi ===")
print(f"Hasil akurasi data train: {acc_train}")
print(f"Hasil akurasi data test: {acc_test}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Laporan Klasifikasi:\n",
      classification_report(y_test, y_pred, target_names=["ham", "spam"],
                            digits=4))


=== Langkah 4: Evaluasi ===
Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9775784753363229
Confusion Matrix:
 [[951   3]
 [ 22 139]]
Laporan Klasifikasi:
               precision    recall  f1-score   support

         ham     0.9774    0.9969    0.9870       954
        spam     0.9789    0.8634    0.9175       161

    accuracy                         0.9776      1115
   macro avg     0.9781    0.9301    0.9523      1115
weighted avg     0.9776    0.9776    0.9770      1115



In [10]:
# Contoh kesalahan klasifikasi
wrong = [(t[:70], "spam" if a else "ham", "spam" if p else "ham")
         for t, a, p in zip(X_test[y_test != y_pred],
                            y_test[y_test != y_pred],
                            y_pred[y_test != y_pred])][:5]
print("\nContoh salah klasifikasi (teks | aktual | prediksi):")
for row in wrong:
    print(" -", " | ".join(row))


Contoh salah klasifikasi (teks | aktual | prediksi):
 - Ever thought about living a good life with a perfect partner? Just txt | spam | ham
 - More people are dogging in your area now. Call 09090204448 and join li | spam | ham
 - How about getting in touch with folks waiting for company? Just txt ba | spam | ham
 - You won't believe it but it's true. It's Incredible Txts! Reply G now  | spam | ham
 - Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. F | spam | ham
